# **Import Library**

> **EXP06 — Revisi dari EXP05.** Perubahan: (1) discriminative LR (backbone kecil, head besar) + warmup-cosine yang diaktifkan lagi, (2) class weight `sqrt(1/count)` bukan `1/count` mentah, (3) EPOCHS diturunkan ke 20, (4) Mixup ringan (alpha=0.2) ditambahkan. Freeze strategy masih `last_4_blocks` -- untuk ablation freeze-depth, jalankan run terpisah dengan `UNFROZEN_LAYERS` diganti manual (`head_only`, `last_2_blocks`, `all`), config lain dibiarkan sama, lalu bandingkan val F1 antar run di W&B.


In [ ]:
# !pip install wandb timm -q

import random
import os
import copy
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import wandb

# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
# wandb.login(key=wandb_api_key)

# Opsi B — pakai .env file (lebih rapi, key nggak ke-expose di kode)
from dotenv import load_dotenv
load_dotenv()  # baca file .env di folder yang sama
wandb.login(key=os.getenv("WANDB_API_KEY"))

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd

from torchvision import transforms, datasets
import timm   # PERBAIKAN: ganti torchvision.models.resnet50 -> timm (untuk ViT)

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# **Dataset Path**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

TRAIN_DIR = r"C:\Users\User\Downloads\Devianest\train"
TEST_DIR  = r"C:\Users\User\Downloads\Devianest\test"

# **Train Augmentation**

In [ ]:
# PERBAIKAN: augmentasi dinaikkan dari "light" -> "medium" sesuai rekomendasi sweep
# (flip + rotasi kecil + color jitter ringan). Untuk skin disease, sengaja TIDAK
# pakai augmentasi "strong" (random crop agresif / cutout / blur) karena bisa
# mengubah ciri visual lesi kulit yang justru jadi fitur penting untuk klasifikasi.
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    #transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    #transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(10),
    #transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05),  # PERBAIKAN: aktifkan, ringan saja
    transforms.RandomResizedCrop(224, scale=(0.8,1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    #transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])


# **Validation Transform**

In [ ]:
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    #transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# **Load Filepaths**

In [ ]:
classes = sorted(os.listdir(TRAIN_DIR))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

num_classes = len(classes)

filepaths = []
labels    = []

for label in classes:
    class_path = os.path.join(TRAIN_DIR, label)
    for img in os.listdir(class_path):
        filepaths.append(os.path.join(class_path, img))
        labels.append(class_to_idx[label])

print("Total Images :", len(filepaths))
print("Classes      :", classes)

# **Dataset Class**

In [ ]:
class SkinDataset(Dataset):

    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# **Early Stopping & K-Fold**

In [ ]:
class EarlyStopping:
    # PERBAIKAN: kriteria checkpoint/early stopping diganti dari val_loss -> val_f1.
    # Aria (WandB AI) menyarankan checkpoint terbaik dipilih dengan kriteria jelas,
    # misalnya best_val_f1 — supaya model yang disimpan benar-benar yang paling
    # bagus performanya (F1), bukan cuma yang val_loss-nya paling rendah (dua hal
    # ini bisa beda, terutama saat data imbalanced).
    def __init__(self, patience=6):
        self.patience  = patience
        self.best_f1   = -np.inf
        self.counter   = 0

    def step(self, val_f1):
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
BATCH_SIZE      = 32
EPOCHS          = 20            # PERBAIKAN: 30 -> 20. Val loss di EXP05 sudah flat sejak epoch ~15-17,
                                 # sisa epoch cuma buang compute (early stopping tetap jadi jaring pengaman).
EXPERIMENT_NAME = "EXP06_ViT_Base_16_warmup_discLR_mixup"

# ── HYPERPARAMETER REVISI (mengatasi temuan dari EXP05) ──────────────────────
# Masalah EXP05: LR=2e-4 kegedean untuk partial fine-tuning ViT pretrained,
# warmup+cosine sudah ditulis tapi di-comment, class weight 1/count kelewat
# ekstrem, dan cuma 1 freeze strategy yang dicoba.
#
# PERBAIKAN 1 — Discriminative LR: backbone (4 block terakhir yang dibuka)
# dilatih pelan (cuma "disesuaikan"), head dilatih lebih cepat (training dari
# scratch, butuh belajar lebih agresif).
BACKBONE_LR      = 2e-5            # utk block yang di-unfreeze + norm
HEAD_LR          = 5e-4            # utk classifier head
WEIGHT_DECAY     = 1e-4
DROP_OUT         = 0.1
UNFROZEN_LAYERS  = "last_4_blocks"  # PERBAIKAN 3 (lihat catatan ablation di bawah):
                                     # ganti manual ke "head_only" / "last_2_blocks" / "all"
                                     # utk run ablation freeze-depth terpisah, config lain dibiarkan sama
AUGMENTATION_STRENGTH = "medium"

# PERBAIKAN 2 — LR warmup + cosine decay DIAKTIFKAN (sebelumnya di-comment).
WARMUP_EPOCHS    = 3

# PERBAIKAN 4 — Mixup ringan (alpha kecil) buat kurangi overfitting tanpa
# merusak fitur lesi individual (beda dari crop/cutout yang berisiko motong
# bagian penting lesi).
USE_MIXUP        = True
MIXUP_ALPHA      = 0.2

run = wandb.init(
    project = "SkinDisease-ViT",
    name    = EXPERIMENT_NAME,
    config  = {
        "architecture"   : "ViT-Base/16 (timm: vit_base_patch16_224)",
        "n_folds"        : 5,
        "epochs"         : EPOCHS,
        "batch_size"     : BATCH_SIZE,
        "optimizer"      : "AdamW",
        "backbone_lr"    : BACKBONE_LR,
        "head_lr"        : HEAD_LR,
        "weight_decay"   : WEIGHT_DECAY,
        "Drop_Out"       : DROP_OUT,
        "unfrozen_layers": UNFROZEN_LAYERS,
        "augmentation_strength": AUGMENTATION_STRENGTH,
        "warmup_epochs"  : WARMUP_EPOCHS,
        "lr_scheduler"   : "linear_warmup_then_cosine_decay",
        "use_mixup"      : USE_MIXUP,
        "mixup_alpha"    : MIXUP_ALPHA,
        "class_weight_fn": "sqrt(1/count)",
        "checkpoint_criteria": "best_val_f1",
    }
)

print(f"WandB Run : {run.name}")
print(f"URL       : {run.url}")
wandb.run.log_code(".")


# **Training Loop**

In [ ]:
# PERBAIKAN: fungsi helper freeze/unfreeze (sama seperti EXP05, tidak diubah)
def apply_freeze_strategy(model, strategy: str):
    for param in model.parameters():
        param.requires_grad = False

    if strategy == "head_only":
        for param in model.head.parameters():
            param.requires_grad = True

    elif strategy == "last_2_blocks":
        for block in model.blocks[-2:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in model.norm.parameters():
            param.requires_grad = True
        for param in model.head.parameters():
            param.requires_grad = True

    elif strategy == "last_4_blocks":
        for block in model.blocks[-4:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in model.norm.parameters():
            param.requires_grad = True
        for param in model.head.parameters():
            param.requires_grad = True

    elif strategy == "all":
        for param in model.parameters():
            param.requires_grad = True

    else:
        raise ValueError(f"Unknown freeze strategy: {strategy}")

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f"  Freeze strategy   : {strategy}")
    print(f"  Trainable params  : {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)")

    return model


# PERBAIKAN 2 — warmup linear + cosine decay, DIAKTIFKAN (dulu di-comment).
# Dipakai per-epoch. Mengembalikan MULTIPLIER (0..1) terhadap base_lr, bukan
# LR absolut, supaya bisa dipakai bareng dua param group (backbone & head)
# yang base_lr-nya beda-beda (discriminative LR).
def get_lr_multiplier(epoch, total_epochs, warmup_epochs):
    import math
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
    return 0.5 * (1 + math.cos(math.pi * progress))


# PERBAIKAN 4 — Mixup ringan. alpha kecil (0.2) -> lam biasanya dekat 0/1,
# jadi campuran gambar tidak terlalu ekstrem, cukup buat regularisasi halus.
def mixup_data(x, y, alpha, device):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=device)
    mixed_x = lam * x + (1 - lam) * x[perm]
    return mixed_x, y, y[perm], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


fold_results = []
fold_accuracies    = []
fold_precision     = []
fold_recall        = []
fold_f1            = []
all_fold_best_paths = []

all_train_losses = {}
all_val_losses   = {}


for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels)):

    print(f"\n{'='*50}")
    print(f"  FOLD {fold + 1} / 5")
    print(f"{'='*50}")

    best_val_loss   = np.inf
    best_train_loss = np.inf
    best_val_f1     = -np.inf
    best_model_path = None

    # ── SPLIT ─────────────────────────────────────────────────────────────────
    train_files  = [filepaths[i] for i in train_idx]
    train_labels = [labels[i]    for i in train_idx]
    val_files    = [filepaths[i] for i in val_idx]
    val_labels   = [labels[i]    for i in val_idx]

    # ── DATALOADER ────────────────────────────────────────────────────────────
    train_loader = DataLoader(
        SkinDataset(train_files, train_labels, transform=train_tf),
        batch_size=BATCH_SIZE, shuffle=True,  num_workers=4,  pin_memory=True
    )
    val_loader = DataLoader(
        SkinDataset(val_files, val_labels, transform=eval_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=4,  pin_memory=True
    )

    # ── MODEL ─────────────────────────────────────────────────────────────────
    model = timm.create_model(
        "vit_base_patch16_224",
        pretrained=True,
        num_classes=num_classes,
        drop_rate=DROP_OUT,
        drop_path_rate=0.1
    )

    model = apply_freeze_strategy(model, UNFROZEN_LAYERS)
    model = model.to(device)

    # ── LOSS ──────────────────────────────────────────────────────────────────
    # PERBAIKAN 3 — class weight: sqrt(1/count) bukan 1/count mentah.
    # Rasio antar kelas jadi jauh lebih halus (misal 6.6x -> ~2.6x), supaya
    # gradient dari kelas minoritas tidak dominan berlebihan tiap batch saat
    # digabung dengan label smoothing + weighted-ness dari class_weights ini.
    class_counts  = np.bincount(train_labels)
    class_weights = torch.sqrt(1. / torch.tensor(class_counts, dtype=torch.float))
    class_weights = class_weights / class_weights.sum() * num_classes  # normalisasi skala

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device), label_smoothing=0.05
    )

    # ── OPTIMIZER — Discriminative LR (PERBAIKAN 1) ──────────────────────────
    # Param group terpisah: backbone (block yang di-unfreeze + norm) pakai
    # BACKBONE_LR kecil, head pakai HEAD_LR lebih besar. filter(requires_grad)
    # tetap dipakai supaya param yang di-freeze tidak ikut masuk optimizer.
    head_params     = list(model.head.parameters())
    head_param_ids  = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters()
                        if p.requires_grad and id(p) not in head_param_ids]

    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": BACKBONE_LR, "name": "backbone"},
        {"params": [p for p in head_params if p.requires_grad], "lr": HEAD_LR, "name": "head"},
    ], weight_decay=WEIGHT_DECAY)

    early_stopping = EarlyStopping(patience=6)
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses = []
    val_losses   = []

    # ── EPOCH LOOP ────────────────────────────────────────────────────────────
    for epoch in range(EPOCHS):

        # PERBAIKAN 2 — terapkan warmup+cosine ke MASING-MASING param group,
        # tiap group tetap pakai base_lr-nya sendiri (BACKBONE_LR / HEAD_LR).
        lr_mult = get_lr_multiplier(epoch, EPOCHS, WARMUP_EPOCHS)
        base_lrs = {"backbone": BACKBONE_LR, "head": HEAD_LR}
        for pg in optimizer.param_groups:
            pg["lr"] = base_lrs[pg["name"]] * lr_mult

        print(f"\nEpoch {epoch + 1}/{EPOCHS} "
              f"(backbone LR: {optimizer.param_groups[0]['lr']:.2e}, "
              f"head LR: {optimizer.param_groups[1]['lr']:.2e})")

        # TRAIN
        model.train()
        train_loss = 0
        for images, targets in tqdm(train_loader, desc="Train"):
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()

            if USE_MIXUP:
                mixed_images, y_a, y_b, lam = mixup_data(images, targets, MIXUP_ALPHA, device)
                outputs = model(mixed_images)
                loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
            else:
                outputs = model(images)
                loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # VALIDATION (tanpa mixup, evaluasi selalu di data asli)
        model.eval()
        val_loss = 0
        preds, trues = [], []
        with torch.no_grad():
            for images, targets in tqdm(val_loader, desc="Val"):
                images, targets = images.to(device), targets.to(device)
                outputs   = model(images)
                val_loss += criterion(outputs, targets).item()
                preds.extend(outputs.argmax(1).cpu().numpy())
                trues.extend(targets.cpu().numpy())

        # METRICS
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss   / len(val_loader)

        acc       = accuracy_score(trues, preds)
        precision = precision_score(trues, preds, average='weighted', zero_division=0)
        recall    = recall_score(trues, preds, average='weighted', zero_division=0)
        f1        = f1_score(trues, preds, average='weighted', zero_division=0)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Train Loss : {avg_train_loss:.4f} | Val Loss  : {avg_val_loss:.4f}")
        print(f"Accuracy   : {acc:.4f}  | Precision : {precision:.4f}")
        print(f"Recall     : {recall:.4f}  | F1 Score  : {f1:.4f}")

        wandb.log({
            "epoch"                    : epoch + 1,

            f"fold_{fold+1}/train_loss": avg_train_loss,
            f"fold_{fold+1}/val_loss"  : avg_val_loss,

            f"fold_{fold+1}/accuracy"  : acc,
            f"fold_{fold+1}/precision" : precision,
            f"fold_{fold+1}/recall"    : recall,
            f"fold_{fold+1}/f1_score"  : f1,

            f"fold_{fold+1}/backbone_lr": optimizer.param_groups[0]['lr'],
            f"fold_{fold+1}/head_lr"    : optimizer.param_groups[1]['lr'],
        })

        # SAVE BEST MODEL (kriteria tetap best_val_f1)
        if f1 > best_val_f1:
            best_val_f1     = f1
            best_val_loss   = avg_val_loss
            best_train_loss = avg_train_loss

            save_path      = f"/kaggle/working/model_fold_{fold + 1}.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_loss"        : avg_val_loss,
                "f1"              : f1,
                "fold"            : fold + 1
            }, save_path)
            best_model_path = save_path
            best_model_wts  = copy.deepcopy(model.state_dict())
            print(f"  ✓ Model saved (best val_f1: {best_val_f1:.4f}) → {save_path}")

        if early_stopping.step(f1):
            print("Early Stopping Triggered")
            break

    # ── SIMPAN HISTORY ────────────────────────────────────────────────────────
    all_train_losses[fold + 1] = train_losses
    all_val_losses[fold + 1]   = val_losses

    # ── PLOT LOSS CURVE PER FOLD ──────────────────────────────────────────────
    epochs_ran = range(1, len(train_losses) + 1)
    fig, ax    = plt.subplots(figsize=(8, 5))
    ax.plot(epochs_ran, train_losses, label='Train Loss', marker='o', markersize=3)
    ax.plot(epochs_ran, val_losses,   label='Val Loss',   marker='o', markersize=3)
    ax.set_title(f'Fold {fold + 1} — Loss Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    curve_path = f"/kaggle/working/Fold_{fold + 1}_Loss_Curve.png"
    fig.savefig(curve_path, dpi=150, bbox_inches='tight')
    wandb.log({f"Loss_Curve/Fold_{fold+1}": wandb.Image(curve_path)})
    plt.close(fig)
    print(f"  ✓ Loss curve saved → {curve_path}")

    # ── UPLOAD MODEL ARTIFACT ─────────────────────────────────────────────────
    if best_model_path:
        artifact = wandb.Artifact(name=f"model-fold-{fold+1}", type="model")
        artifact.add_file(best_model_path)
        wandb.log_artifact(artifact)
        all_fold_best_paths.append(best_model_path)

    # ── FINAL EVALUATION FOLD (pakai best model) ──────────────────────────────
    if best_model_path:
        model.load_state_dict(
            torch.load(best_model_path, map_location=device)["model_state_dict"]
        )

    model.eval()
    final_preds, final_trues = [], []
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            final_preds.extend(model(images).argmax(1).cpu().numpy())
            final_trues.extend(targets.cpu().numpy())

    print("\nClassification Report")
    print(classification_report(final_trues, final_preds, target_names=classes, zero_division=0))

    fold_acc  = accuracy_score(final_trues, final_preds)
    fold_prec = precision_score(final_trues, final_preds, average='weighted', zero_division=0)
    fold_rec  = recall_score(final_trues, final_preds, average='weighted', zero_division=0)
    fold_f1_  = f1_score(final_trues, final_preds, average='weighted', zero_division=0)

    fold_accuracies.append(fold_acc)
    fold_precision.append(fold_prec)
    fold_recall.append(fold_rec)
    fold_f1.append(fold_f1_)

    fold_results.append({
        "Fold": fold + 1,
        "Train_Loss": best_train_loss,
        "Val_Loss": best_val_loss,
        "Accuracy": fold_acc,
        "Precision": fold_prec,
        "Recall": fold_rec,
        "F1": fold_f1_
})

    wandb.log({
        f"fold_{fold+1}/final_accuracy" : fold_acc,
        f"fold_{fold+1}/final_precision": fold_prec,
        f"fold_{fold+1}/final_recall"   : fold_rec,
        f"fold_{fold+1}/final_f1"       : fold_f1_,

        f"fold_{fold+1}/confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=final_trues,
            preds=final_preds,
            class_names=classes
        )
    })

    print(f"\nFold {fold+1} selesai — Acc: {fold_acc:.4f} | F1: {fold_f1_:.4f}")

    del model, optimizer, best_model_wts
    torch.cuda.empty_cache()


results_df = pd.DataFrame(fold_results)

results_df.loc[len(results_df)] = {
    "Fold": "Mean",
    "Train_Loss": results_df["Train_Loss"].mean(),
    "Val_Loss": results_df["Val_Loss"].mean(),
    "Accuracy": np.mean(fold_accuracies),
    "Precision": np.mean(fold_precision),
    "Recall": np.mean(fold_recall),
    "F1": np.mean(fold_f1)
}

csv_path = "/kaggle/working/KFold_Summary.csv"
results_df.to_csv(csv_path, index=False)

artifact = wandb.Artifact(
    "kfold-summary",
    type="results"
)

artifact.add_file(csv_path)

wandb.log_artifact(artifact)


# **Grafik Gabungan & Final Summary**

In [ ]:
# ── GRAFIK GABUNGAN SEMUA FOLD ────────────────────────────────────────────────
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, fold_n in enumerate(all_train_losses.keys()):
    ep = range(1, len(all_train_losses[fold_n]) + 1)
    c  = colors[(fold_n - 1) % len(colors)]
    axes[0].plot(ep, all_train_losses[fold_n], label=f'Fold {fold_n}', color=c, marker='o', markersize=3)
    axes[1].plot(ep, all_val_losses[fold_n],   label=f'Fold {fold_n}', color=c, marker='o', markersize=3)

for ax, title in zip(axes, ['Train Loss — Semua Fold', 'Val Loss — Semua Fold']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Perbandingan Loss Semua Fold', fontsize=14, fontweight='bold')
plt.tight_layout()

combined_path = "/kaggle/working/All_Folds_Loss_Curve.png"
fig.savefig(combined_path, dpi=150, bbox_inches='tight')
wandb.log({"Loss_Curve/All_Folds_Combined": wandb.Image(combined_path)})
plt.close(fig)
print(f"✓ Grafik gabungan disimpan → {combined_path}")

# ── SUMMARY METRICS ───────────────────────────────────────────────────────────
print("\n" + "="*50)
print("  FINAL RESULT — ALL FOLDS")
print("="*50)
print(f"Mean Accuracy  : {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")
print(f"Mean Precision : {np.mean(fold_precision):.4f} ± {np.std(fold_precision):.4f}")
print(f"Mean Recall    : {np.mean(fold_recall):.4f} ± {np.std(fold_recall):.4f}")
print(f"Mean F1 Score  : {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")

# ── WANDB LOG SUMMARY ─────────────────────────────────────────────────────────
# Panel Summary → summary/mean_accuracy, summary/mean_precision, dst.
wandb.log({
    "summary/mean_accuracy"  : np.mean(fold_accuracies),
    "summary/mean_precision" : np.mean(fold_precision),
    "summary/mean_recall"    : np.mean(fold_recall),
    "summary/mean_f1"        : np.mean(fold_f1),
    "summary/std_accuracy"   : np.std(fold_accuracies),
    "summary/std_f1"         : np.std(fold_f1),
})



# **Test Evaluation**

In [ ]:
best_overall_path = all_fold_best_paths[fold_f1.index(max(fold_f1))]
print(f"Best model path : {best_overall_path}")
print(f"Best F1         : {max(fold_f1):.4f}")

test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_tf)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# PERBAIKAN: di versi ResNet50, variabel `model` yang dipakai di sini adalah sisa
# `model` dari iterasi fold terakhir Cell 17 (kebetulan arsitekturnya sama tiap
# fold, jadi tidak error, tapi rapuh). Di sini dibuat eksplisit: instance model
# ViT-Base baru, lalu load bobot terbaik -> lebih jelas & tidak tergantung state
# sisa loop sebelumnya.
model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=num_classes
).to(device)

checkpoint = torch.load(best_overall_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


# **Test Confusion Matrix**

In [ ]:
y_true, y_pred = [], []

with torch.no_grad():
    for images, lbs in tqdm(test_loader, desc="Test"):
        images  = images.to(device)
        outputs = model(images)
        y_true.extend(lbs.cpu().numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())

acc       = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm  = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(15, 15))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(
    cmap='Blues', ax=ax, xticks_rotation=90
)
plt.tight_layout()
plt.savefig("/kaggle/working/Test_Confusion_Matrix.png", dpi=150, bbox_inches='tight')
plt.show()

wandb.log({
    "Test/Confusion_Matrix": wandb.Image(
        "/kaggle/working/Test_Confusion_Matrix.png"
    ),
    "test/accuracy": acc,
    "test/precision": precision,
    "test/recall": recall,
    "test/f1": f1
})

# ==========================================
# TEST RESULT CSV
# ==========================================

test_results_df = pd.DataFrame({
    "Filename": [test_dataset.samples[i][0] for i in range(len(y_true))],
    "True_Label": [classes[i] for i in y_true],
    "Predicted_Label": [classes[i] for i in y_pred],
    "Correct": np.array(y_true) == np.array(y_pred)
})


test_summary_df = pd.DataFrame([{
    "Accuracy": acc,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
}])

summary_path = "/kaggle/working/Test_Summary.csv"
test_summary_df.to_csv(summary_path, index=False)

csv_test_path = "/kaggle/working/Test_Result.csv"
test_results_df.to_csv(csv_test_path, index=False)

print(f"Test CSV saved -> {csv_test_path}")


# ==========================================
# UPLOAD TEST CSV KE WANDB
# ==========================================

artifact = wandb.Artifact(
    name="test-results",
    type="results"
)

artifact.add_file(csv_test_path)
artifact.add_file(summary_path)

wandb.log_artifact(artifact)

wandb.finish()
print("\nWandB run selesai.")